<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex07.1-poisson/Ex07.1_02_slot_hard_bc.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_07.1 · Notebook 02 — The Wall Condition **Built In**

**Paired with L7.1 · Fundamentals of PINNs**

Notebook 01 asked the network to satisfy the walls. This one makes it
impossible to do otherwise.

Write the solution as a product:

$$\hat\theta(x,y) \;=\; \underbrace{(1-\xi^2)(1-\eta^2)}_{D(x,y)} \; N(x,y),
\qquad \xi = x/a, \ \eta = y/b$$

$D$ vanishes on all four walls, so $\hat\theta$ does too — for **any** network
$N$, at every stage of training, to machine precision. The boundary condition
is no longer something the optimiser trades against; it is a property of the
function space.

The loss loses a term and a hyperparameter:

$$\mathcal{L} = \frac{1}{N_f}\sum \bigl(k_{\mathrm{eff}}\nabla^2\hat\theta + q\bigr)^2$$

## What you will do

1. Build the trial solution and confirm the walls are exact before training.
2. Train with one loss term and no weight to choose.
3. Compare against notebook 01 on the same grid.
4. Find where the method stops being easy.

---

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex07.1-poisson/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

## 1 · The trial solution

`pb.hard_bc_factor(x, y)` is $D$. Confirm it does what is claimed **before**
training anything — the whole argument rests on it.

### Your turn

In [ ]:
# TODO: build the trial solution and check it on the walls.
#
#   def theta_trial(model, xy):
#       x, y = xy[:, 0:1], xy[:, 1:2]
#       return pb.hard_bc_factor(x, y) * model(xy)
#
#   Then, with an UNTRAINED network:
#       set_seed(0)
#       probe = MLP(n_in=2, n_hidden=32, n_layers=4)
#       wall  = to_tensor(boundary_points(200, pb.DOMAIN, seed=3))
#       with torch.no_grad():
#           on_wall = to_numpy(theta_trial(probe, wall))
#   and record wall_untrained = np.abs(on_wall).max()

raise NotImplementedError("Write theta_trial and check it on the walls")

In [ ]:
print(f"worst wall error, untrained network : {wall_untrained:.3e} K")
print(f"notebook 01 needed training to reach : ~1e-2 K, and never zero")
check("walls are exact before any training", wall_untrained, 0.0, tol=1e-14)

**What you should see.** A `PASS`, with an error of exactly zero or at worst a
few times 1e-17.

Sit with that for a second. The network is random — it has been told nothing
about heat, slots or walls — and the boundary condition already holds to
machine precision. Notebook 01 spent three thousand Adam steps and a weight
sweep to get within a hundredth of a kelvin.

---

## 2 · The residual and the loss

Same PDE as before, but now applied to the *trial solution* rather than to the
network output. There is no boundary term.

### Your turn

In [ ]:
# TODO: residual and loss for the hard-enforcement model.
#
#   def pde_residual_hard(model, xy):
#       theta = theta_trial(model, xy)
#       return pb.K_EFF * (d2(theta, xy, 0) + d2(theta, xy, 1)) \
#              + pb.source(xy[:, 0:1], xy[:, 1:2])
#
#   Q_SCALE = float(pb.source(np.array([0.0]), np.array([0.0])))
#
#   def make_loss_hard(model, xy_f):
#       def loss():
#           return mse(pde_residual_hard(model, xy_f) / Q_SCALE)
#       return loss
#
# Note that d2 now differentiates the PRODUCT D * N, so the chain rule brings
# in derivatives of D as well. You do not have to write those -- autograd does
# -- but it is worth knowing they are there, because they are the reason the
# residual is a little more expensive here than in notebook 01.

raise NotImplementedError("Write pde_residual_hard and make_loss_hard")

## 3 · Train

In [ ]:
N_COLL = 2000

set_seed(88)
model_hard = MLP(n_in=2, n_hidden=32, n_layers=4)
describe(model_hard, N_COLL)

xy_f = to_tensor(interior_points(N_COLL, pb.DOMAIN, seed=1), requires_grad=True)

history_hard = train_two_stage(model_hard, make_loss_hard(model_hard, xy_f),
                               adam_steps=3000, lbfgs_steps=150, lr=1e-3)
plot_curves(history_hard, title="hard enforcement — one term, no weight")
plt.show()

## 4 · The comparison

Same grid, same metrics, same seed. Notebook 01's results are loaded from disk
so nothing is retyped.

### Your turn

In [ ]:
# TODO: score the hard model and put it beside the soft one.
#
#   X, Y, pts = grid_points(161, 161, pb.DOMAIN)
#   with torch.no_grad():
#       theta_hard = to_numpy(theta_trial(model_hard, to_tensor(pts))).ravel()
#   theta_ref = pb.theta_exact(pts[:, 0], pts[:, 1])
#
#   rel_hard  = relative_l2(theta_hard, theta_ref)
#   max_hard  = max_abs_error(theta_hard, theta_ref)
#
#   wall = to_tensor(boundary_points(200, pb.DOMAIN, seed=7))
#   with torch.no_grad():
#       wall_hard = float(np.abs(to_numpy(theta_trial(model_hard, wall))).max())
#
#   Load notebook 01 with:
#       soft = np.load(os.path.join("Ex07.1_outputs", "nb01_soft.npz"))

raise NotImplementedError("Score the hard-enforcement model and compare")

In [ ]:
print(error_table(
    [["soft, w = 1", f"{float(soft['rel']):.3e}", f"{float(soft['max_err']):.4f}",
      f"{float(soft['wall']):.2e}", "yes"],
     ["hard, no weight", f"{rel_hard:.3e}", f"{max_hard:.4f}",
      f"{wall_hard:.2e}", "no"]],
    ["enforcement", "relative L2", "worst error [K]", "worst wall [K]",
     "weight to tune?"]))

fig, axes = plt.subplots(1, 3, figsize=(14.0, 6.0))
pb.plot_field(theta_ref, ax=axes[0], title="exact")
pb.plot_error(soft["theta_hat"], ax=axes[1], title="soft — signed error [K]")
pb.plot_error(theta_hard, ax=axes[2], title="hard — signed error [K]")
plt.tight_layout(); plt.show()

**What you should see.** The wall error drops by ten or more orders of
magnitude — from about 1e-2 K to about 1e-17 K, which is zero. The field error
usually improves too, often by a factor of a few.

And the column that matters most is the last one. Hard enforcement removed a
hyperparameter. You are no longer choosing a number that you could only have
chosen honestly by knowing the answer.

Look at the two error maps together. The soft model's error tends to pile up
near the walls; the hard model's cannot, so whatever error remains sits in the
interior where the physics is, which is a more useful place for it to be.

---

## 5 · Where this stops being easy

Hard enforcement is not free, and L7.1 says so. Four honest limits:

**It needs a $D$ you can write down.** A rectangle is trivial. An L-shaped slot,
a plate with a hole, an aerofoil — constructing a smooth function that vanishes
on exactly that boundary and nowhere else is a real problem, and it is why
Ex_08.1 keeps a soft term for the hole.

**It hard-codes the boundary *value*, not just the location.** Here the walls
are at zero excess temperature, so $D \cdot N$ suffices. For a non-zero or
varying wall temperature you need $\hat\theta = g(x,y) + D \cdot N$, with $g$ a
smooth function matching the data on the boundary — and inventing $g$ can be
harder than the original problem.

**Neumann and Robin conditions do not factor this way.** A prescribed *flux* is
a condition on the derivative, and there is no factor that makes it hold
automatically. Those stay soft.

**$D$ changes the conditioning.** The trial solution is forced to zero at the
walls, so its gradients near them are dominated by $D$'s. On a thin domain —
and a slot is thin — that can slow training even while it makes the boundary
exact.

### Your turn

In [ ]:
# TODO: demonstrate the second limit for yourself.
#
#   Suppose the slot walls are NOT at a uniform temperature: the closed end
#   (y = -b) sits 8 K hotter than the other three, because the iron is thicker
#   there and carries less flux away.
#
#   Write a lift function g(x, y) that is smooth on the slot and matches
#       g = 8 K   on y = -b
#       g = 0     on the other three walls
#   A workable choice:  g(x, y) = 8 * (1 - eta) / 2 * (1 - xi**2)
#       -- check it yourself on all four walls before believing it.
#
#   Then confirm  theta = g + D * N  satisfies all four wall conditions for an
#   untrained N, and record the four worst errors in lift_check = {...}.
#
# You are not asked to train this one. The point is to feel how much work the
# construction is for a shape this simple.

raise NotImplementedError("Construct and check the lift function g")

In [ ]:
for wall_name, err in lift_check.items():
    print(f"  {wall_name:<8s} worst error {err:.2e} K")
print()
print("Now imagine doing that for a slot with a rounded closed end.")

## 6 · Save

In [ ]:
path = os.path.join("Ex07.1_outputs", "nb02_hard.npz")
np.savez(path,
         rel=rel_hard, max_err=max_hard, wall=wall_hard,
         theta_hat=theta_hard,
         adam=history_hard["adam"], lbfgs=history_hard["lbfgs"])
torch.save(model_hard.state_dict(), os.path.join("Ex07.1_outputs", "nb02_hard.pt"))
print("wrote", path)

## 7 · Before you move on

1. The untrained network already satisfied the walls exactly. Say precisely
   why, in terms of the trial solution — not "because of the factor".
2. Hard enforcement removed a hyperparameter. What did it add?
3. Ex_08.1 solves a plate with a hole and keeps the hole's condition **soft**.
   Predict why, then check your prediction when you get there.
4. Which of the two methods would you use for a prescribed heat flux, and what
   goes wrong with the other?

Next: **notebook 03**, the comparison and the report.